**Create BigQuery Table**

In [1]:
from google.cloud import bigquery
from google.oauth2 import service_account

SERVICE_ACCOUNT_FILE = "../Keys/BigQueryKey.json"

credentials = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE
)

PROJECT_ID = "depihealthnux"
DATASET_ID = "Depihealthnux_Main"
TABLE_ID = "Dr_Time_Table"

TABLE_REF = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}"

client = bigquery.Client(
    credentials=credentials,
    project=PROJECT_ID
)

schema = [

    bigquery.SchemaField(
        "Time_Table_Key",
        "STRING",
        mode="REQUIRED"
    ),

    bigquery.SchemaField(
        "Dr_Code",
        "STRING"
    ),

    bigquery.SchemaField(
        "Speciality",
        "STRING"
    ),

    bigquery.SchemaField(
        "Dr_Name",
        "STRING"
    ),

    bigquery.SchemaField(
        "Day_of_Week",
        "STRING"
    ),

    bigquery.SchemaField(
        "Scheduled_Start_Time",
        "TIME"
    ),

    bigquery.SchemaField(
        "Scheduled_End_Time",
        "TIME"
    ),

    bigquery.SchemaField(
        "Capacity",
        "INTEGER"
    ),

    bigquery.SchemaField(
        "Is_Active",
        "STRING"
    )

]

table = bigquery.Table(
    TABLE_REF,
    schema=schema
)

table.clustering_fields = [
    "Dr_Code",
    "Day_of_Week"
]

client.create_table(
    table,
    exists_ok=True
)

print("=================================")
print("BigQuery table created successfully")
print(TABLE_REF)
print("=================================")

BigQuery table created successfully
depihealthnux.Depihealthnux_Main.Dr_Time_Table


**Create Postgres Table**

In [2]:
import sys
import psycopg2

sys.path.append("..")
from Keys.PostGresKey import POSTGRES_URL

conn = psycopg2.connect(POSTGRES_URL)
cursor = conn.cursor()

cursor.execute("""

CREATE SEQUENCE IF NOT EXISTS timetable_no_seq;

""")

create_table_query = """
CREATE TABLE IF NOT EXISTS Dr_Time_Table (

    Time_Table_Key TEXT PRIMARY KEY
    DEFAULT (
        'TT' ||
        LPAD(
            nextval('timetable_no_seq')::TEXT,
            4,
            '0'
        )
    ),

    Dr_Code TEXT NOT NULL,

    Day_of_Week TEXT,

    Scheduled_Start_Time TIME,

    Scheduled_End_Time TIME,

    Capacity INTEGER,

    Is_Active TEXT

);
"""

cursor.execute(create_table_query)

cursor.execute("""

CREATE INDEX IF NOT EXISTS idx_tt_dr_code
ON Dr_Time_Table(Dr_Code);

""")

cursor.execute("""

CREATE INDEX IF NOT EXISTS idx_tt_day
ON Dr_Time_Table(Day_of_Week);

""")

conn.commit()

print("=================================")
print("PostgreSQL table created successfully")
print("Table: Dr_Time_Table")
print("=================================")

cursor.close()
conn.close()

PostgreSQL table created successfully
Table: Dr_Time_Table


**Sync BigQuery to Postgres**

In [3]:
import sys
import pandas as pd
import psycopg2

from psycopg2.extras import execute_values

from google.cloud import bigquery
from google.oauth2 import service_account

sys.path.append("..")
from Keys.PostGresKey import POSTGRES_URL

credentials = service_account.Credentials.from_service_account_file(
    "../Keys/BigQueryKey.json"
)

client = bigquery.Client(
    credentials=credentials,
    project="depihealthnux"
)

query = """

SELECT

    Time_Table_Key,
    Dr_Code,
    Day_of_Week,
    Scheduled_Start_Time,
    Scheduled_End_Time,
    Capacity,
    Is_Active

FROM `depihealthnux.Depihealthnux_Main.Dr_Time_Table`

ORDER BY Time_Table_Key

"""

df = client.query(query).to_dataframe()

print(df.head(3))
print(f"Rows Retrieved: {len(df)}")

conn = psycopg2.connect(POSTGRES_URL)
cursor = conn.cursor()

cursor.execute("""
DELETE FROM Dr_Time_Table;
""")

if not df.empty:

    execute_values(
        cursor,
        """
        INSERT INTO Dr_Time_Table (

            Time_Table_Key,
            Dr_Code,
            Day_of_Week,
            Scheduled_Start_Time,
            Scheduled_End_Time,
            Capacity,
            Is_Active

        )
        VALUES %s
        """,
        df.values.tolist(),
        page_size=1000
    )

cursor.execute("""

SELECT setval(
    'timetable_no_seq',
    COALESCE(
        (
            SELECT MAX(
                CAST(
                    REPLACE(Time_Table_Key,'TT','')
                    AS INTEGER
                )
            )
            FROM Dr_Time_Table
        ),
        1
    ),
    true
);

""")

conn.commit()

print(f"Inserted {len(df)} rows")

cursor.execute("""

SELECT *
FROM Dr_Time_Table
ORDER BY Time_Table_Key
LIMIT 3

""")

print("\nFirst 3 Rows From PostgreSQL")

for row in cursor.fetchall():
    print(row)

cursor.close()
conn.close()

Empty DataFrame
Columns: [Time_Table_Key, Dr_Code, Day_of_Week, Scheduled_Start_Time, Scheduled_End_Time, Capacity, Is_Active]
Index: []
Rows Retrieved: 0
Inserted 0 rows

First 3 Rows From PostgreSQL


**Sync Postgres to BigQuery**

In [4]:
import sys
import pandas as pd
import psycopg2

from google.cloud import bigquery
from google.oauth2 import service_account

sys.path.append("..")
from Keys.PostGresKey import POSTGRES_URL

credentials = service_account.Credentials.from_service_account_file(
    "../Keys/BigQueryKey.json"
)

client = bigquery.Client(
    credentials=credentials,
    project="depihealthnux"
)

conn = psycopg2.connect(POSTGRES_URL)

query = """

SELECT

    t.Time_Table_Key,

    t.Dr_Code,

    d.Speciality,

    d.Dr_Name,

    t.Day_of_Week,

    t.Scheduled_Start_Time,

    t.Scheduled_End_Time,

    t.Capacity,

    t.Is_Active

FROM Dr_Time_Table t

LEFT JOIN Dr_List d
    ON t.Dr_Code = d.Dr_Code

ORDER BY t.Time_Table_Key

"""

df = pd.read_sql(query, conn)

print(df.head(3))
print(f"Rows Retrieved: {len(df)}")

conn.close()

job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE"
)

job = client.load_table_from_dataframe(
    df,
    "depihealthnux.Depihealthnux_Main.Dr_Time_Table",
    job_config=job_config
)

job.result()

print(f"Uploaded {len(df)} rows")

verify_df = client.query("""

SELECT *
FROM `depihealthnux.Depihealthnux_Main.Dr_Time_Table`
LIMIT 3

""").to_dataframe()

print("\nFirst 3 Rows From BigQuery")

print(verify_df)

C:\Users\Sedeek Elmasry\AppData\Local\Temp\ipykernel_1912\1351373951.py:53: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Empty DataFrame
Columns: [time_table_key, dr_code, speciality, dr_name, day_of_week, scheduled_start_time, scheduled_end_time, capacity, is_active]
Index: []
Rows Retrieved: 0
Uploaded 0 rows

First 3 Rows From BigQuery
Empty DataFrame
Columns: [time_table_key, dr_code, speciality, dr_name, day_of_week, scheduled_start_time, scheduled_end_time, capacity, is_active]
Index: []
